# Recursive tree map

Developing a treemap optimization template. Testing tree: the file/directory tree of `src/vizopt` itself.

In [ ]:
import pathlib

import networkx as nx
from matplotlib import pyplot as plt

from vizopt import introspection

In [ ]:
src_dir = pathlib.Path().resolve().parents[1] / "src" / "vizopt"
src_dir

In [ ]:
file_tree = introspection.build_file_tree(src_dir)
nx.is_arborescence(file_tree), file_tree.number_of_nodes(), file_tree.number_of_edges()

## Baseline: squarified treemap heuristic

`introspection.plot_treemap` already gives a deterministic (non-optimized) layout via `squarify_layout`, using each file's byte size as its weight. Useful as a reference/initializer for the new template.

In [ ]:
introspection.plot_treemap(file_tree, padding=0.025)

## `RasterTreemapOptimizer`

Recursive raster-based star-domain treemap, promoted to `vizopt.templates.trees.recursive_raster_treemap.RasterTreemapOptimizer` (with unit tests in `tests/test_recursive_raster_treemap.py`) after developing it interactively against this tree: each node's children are jointly fit with `RasterStarOptimizer` (raster collision for mutual exclusion, an analytic containment term against the node's own already-fitted boundary, and a compactness term to close the whitespace exclusion+area+perimeter alone would leave behind), then recursed into depth-first using each child's *achieved* area as the next level's container budget.

Two generalizations made while extracting it:

- `star_polygon_area` and `radius_at_angle` moved to `vizopt.components.stars` as public numpy utilities — the same area formula was duplicated inline in this exploration, in `raster_based.ipynb`'s British Isles cell, and in the JAX `_multi_term_area` term.
- Child-rectangle seeding now calls `vizopt.treemap.squarify_layout` directly instead of `introspection.treemap_layout`, since the latter is file-tree-specific (it branches on a `graph.nodes[child]["is_dir"]` attribute that only `build_file_tree` graphs carry). The module works on any tree given a `sizes` dict, not just file trees.

In [ ]:
from vizopt.templates.trees.recursive_raster_treemap import RasterTreemapOptimizer

sizes = introspection.compute_subtree_sizes(file_tree)

treemap_optimizer = RasterTreemapOptimizer(
    file_tree,
    sizes,
    grid_resolution=96,
    n_iters=2500,
    learning_rate=0.01,
)
treemap_optimizer.optimize()
len(treemap_optimizer.node_shapes_)

In [ ]:
ax = treemap_optimizer.plot()
ax.set_title("Recursive raster treemap of src/vizopt (RasterTreemapOptimizer)")
plt.gcf().set_size_inches(10, 10)
plt.show()